In [6]:
!nvidia-smi

Sun May 10 16:27:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from google.colab import drive
drive.mount('/content/drive')

# Vérifier l'accès
import os
PROJECT_DIR = "/content/drive/MyDrive/chat_musicien"
assert os.path.exists(PROJECT_DIR), f"Dossier introuvable : {PROJECT_DIR}"
print("✅ Drive monté, projet trouvé")

for sub in ["data", "src", "models"]:
    p = f"{PROJECT_DIR}/{sub}"
    print(f"  {sub}/ : {len(os.listdir(p)) if os.path.exists(p) else 'MANQUE'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive monté, projet trouvé
  data/ : 3
  src/ : 5
  models/ : 0


In [8]:
import sys
sys.path.insert(0, PROJECT_DIR)   # pour que `from src...` marche

from src.gpt import GPTModel, GPT_CONFIG_MUSIC
from src.dataset import make_dataloaders

print(f"Config : {GPT_CONFIG_MUSIC}")

Config : {'vocab_size': 157, 'context_length': 256, 'emb_dim': 256, 'n_heads': 8, 'n_layers': 6, 'drop_rate': 0.1, 'qkv_bias': False}


In [9]:
import pickle
TRAIN_PKL = f"{PROJECT_DIR}/data/train_tokens.pkl"
VAL_PKL   = f"{PROJECT_DIR}/data/val_tokens.pkl"

with open(TRAIN_PKL, "rb") as f:
    train_seqs = pickle.load(f)
with open(VAL_PKL, "rb") as f:
    val_seqs = pickle.load(f)

print(f"Train : {len(train_seqs):,} morceaux | total {sum(len(s) for s in train_seqs):,} tokens")
print(f"Val   : {len(val_seqs):,} morceaux | total {sum(len(s) for s in val_seqs):,} tokens")

Train : 10,000 morceaux | total 84,720,442 tokens
Val   : 526 morceaux | total 4,373,762 tokens


In [10]:
import time, torch

# Petit benchmark pour estimer la vitesse réelle
class BenchArgs:
    train_pkl   = TRAIN_PKL
    val_pkl     = VAL_PKL
    output_dir  = f"{PROJECT_DIR}/models"
    device      = "cuda"
    epochs      = 1
    steps       = 100               # ← test 100 steps seulement
    batch_size  = 64
    stride      = 128
    lr          = 3e-4
    num_workers = 2
    log_every   = 20
    eval_every  = 10000             # ← pas d'éval pendant le bench
    resume      = ""

from src.train import train
t0 = time.time()
train(BenchArgs())
elapsed = time.time() - t0
print(f"\n⏱️ 100 steps en {elapsed:.1f}s → {elapsed/100*1000:.0f} ms/step")
print(f"   Pour 1 epoch ({10342} steps) : ~{(elapsed/100)*10342/60:.1f} min")
print(f"   Pour 3 epochs : ~{(elapsed/100)*10342*3/60:.0f} min ({(elapsed/100)*10342*3/3600:.1f} h)")

Device : cuda
GPU : Tesla T4
Train : 661,877 fenêtres  |  Val : 34,169 fenêtres
Batches / epoch : 10,341
Paramètres : 4,880,384 (~4.88 M)
epoch 1/1 | step 20 | loss 4.1130 | lr 2.82e-04 | 8.0s
epoch 1/1 | step 40 | loss 3.3800 | lr 2.10e-04 | 14.9s
epoch 1/1 | step 60 | loss 3.2715 | lr 1.13e-04 | 21.8s
epoch 1/1 | step 80 | loss 3.2082 | lr 3.16e-05 | 28.8s
epoch 1/1 | step 100 | loss 3.1335 | lr 0.00e+00 | 35.8s

✅ Entraînement terminé en 0.6 min
   best val loss : inf  |  perplexity : 485165195.41
   modèle sauvegardé dans /content/drive/MyDrive/chat_musicien/models

⏱️ 100 steps en 53.3s → 533 ms/step
   Pour 1 epoch (10342 steps) : ~91.9 min
   Pour 3 epochs : ~276 min (4.6 h)


In [11]:
# ENTRAÎNEMENT COMPLET DU CHAT MUSICIEN
# Durée estimée : ~3h sur T4
# Sauvegardes plus fréquentes pour sécuriser

class TrainArgs:
    train_pkl   = TRAIN_PKL
    val_pkl     = VAL_PKL
    output_dir  = f"{PROJECT_DIR}/models"
    device      = "cuda"
    epochs      = 2
    steps       = 0
    batch_size  = 64
    stride      = 128
    lr          = 3e-4
    num_workers = 2
    log_every   = 200
    eval_every  = 500                # ← Plus fréquent (sécurité)
    resume      = ""

from src.train import train
train(TrainArgs())

Device : cuda
GPU : Tesla T4
Train : 661,877 fenêtres  |  Val : 34,169 fenêtres
Batches / epoch : 10,341
Paramètres : 4,880,384 (~4.88 M)
epoch 1/2 | step 200 | loss 3.1759 | lr 1.20e-04 | 80.1s
epoch 1/2 | step 400 | loss 2.9314 | lr 2.40e-04 | 158.9s
  >> val_loss 2.8146 | perplexity 16.69
  ✅ nouveau best, sauvegardé
epoch 1/2 | step 600 | loss 2.6966 | lr 3.00e-04 | 244.4s
epoch 1/2 | step 800 | loss 2.5477 | lr 3.00e-04 | 322.2s
epoch 1/2 | step 1000 | loss 2.4275 | lr 3.00e-04 | 399.8s
  >> val_loss 2.4665 | perplexity 11.78
  ✅ nouveau best, sauvegardé
epoch 1/2 | step 1200 | loss 2.4425 | lr 2.99e-04 | 484.1s
epoch 1/2 | step 1400 | loss 2.4264 | lr 2.99e-04 | 561.2s
  >> val_loss 2.3834 | perplexity 10.84
  ✅ nouveau best, sauvegardé
epoch 1/2 | step 1600 | loss 2.4779 | lr 2.98e-04 | 645.5s
epoch 1/2 | step 1800 | loss 2.3828 | lr 2.97e-04 | 723.7s
epoch 1/2 | step 2000 | loss 2.3055 | lr 2.96e-04 | 802.4s
  >> val_loss 2.3292 | perplexity 10.27
  ✅ nouveau best, sauvegardé
e